# agentic-civic-resolution-app 
# Day 1 | Notebook 2: Bronze → Silver Pipeline
**Goal:** Clean, type-cast, enrich, and deduplicate Bronze data → Silver Delta table

Silver layer guarantees:
Typed columns (dates, floats, booleans)
Deduplicated on `unique_key`
Null-safe complaint category & status
Derived fields: `days_open`, `is_closed`, `borough_clean`

## 0. Config

In [0]:
CATALOG       = "civicops"
BRONZE_TABLE  = f"{CATALOG}.bronze.nyc311_raw"
SILVER_TABLE  = f"{CATALOG}.silver.complaints"


## 1. Load Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

bronze = spark.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze.count():,}")
display(bronze.limit(3))


## 2. Select & Cast Core Columns

In [0]:
silver = (
    bronze
    # ── Identifiers ──────────────────────────────────────────────
    .withColumn("complaint_id",
        F.col("unique_key").cast(T.StringType()))

    # ── Dates ────────────────────────────────────────────────────
    .withColumn("created_date",
        F.to_timestamp("created_date", "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn("closed_date",
        F.to_timestamp("closed_date",  "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn("due_date",
        F.to_timestamp("due_date",     "yyyy-MM-dd'T'HH:mm:ss.SSS"))

    # ── Complaint Details ─────────────────────────────────────────
    .withColumn("complaint_type",
        F.upper(F.trim(F.col("complaint_type"))))
    .withColumn("descriptor",
        F.trim(F.col("descriptor")))
    .withColumn("status",
        F.upper(F.trim(F.col("status"))))
    .withColumn("resolution_description",
        F.col("resolution_description").cast(T.StringType()))

    # ── Agency ────────────────────────────────────────────────────
    .withColumn("agency",       F.col("agency").cast(T.StringType()))
    .withColumn("agency_name",  F.col("agency_name").cast(T.StringType()))

    # ── Location ─────────────────────────────────────────────────
    .withColumn("borough",
        F.upper(F.trim(F.col("borough"))))
    .withColumn("borough_clean",
        F.when(F.col("borough") == "UNSPECIFIED", None).otherwise(F.col("borough")))
    .withColumn("city",        F.trim(F.col("city")).cast(T.StringType()))
    .withColumn("zip_code",    F.col("incident_zip").cast(T.StringType()))
    .withColumn("street_name", F.col("street_name").cast(T.StringType()))
    .withColumn("latitude",
        F.col("latitude").cast(T.DoubleType()))
    .withColumn("longitude",
        F.col("longitude").cast(T.DoubleType()))
    .withColumn("has_geo",
        F.col("latitude").isNotNull() & F.col("longitude").isNotNull())

    # ── Derived / Enriched ────────────────────────────────────────
    .withColumn("is_closed",
        F.col("status").isin("CLOSED", "CLOSE"))
    .withColumn("days_open",
        F.when(
            F.col("is_closed") & F.col("closed_date").isNotNull(),
            F.datediff(F.col("closed_date"), F.col("created_date"))
        ).otherwise(
            F.datediff(F.current_date(), F.col("created_date"))
        ).cast(T.IntegerType()))
    .withColumn("open_year",  F.year("created_date").cast(T.IntegerType()))
    .withColumn("open_month", F.month("created_date").cast(T.IntegerType()))
    .withColumn("open_dow",   F.dayofweek("created_date").cast(T.IntegerType()))

    # ── Metadata ──────────────────────────────────────────────────
    .withColumn("_silver_at", F.current_timestamp())
)


## 3. Select Final Silver Columns

In [0]:
SILVER_COLS = [
    # Identifiers
    "complaint_id",
    # Dates
    "created_date", "closed_date", "due_date",
    # Complaint details
    "complaint_type", "descriptor", "status", "resolution_description",
    # Agency
    "agency", "agency_name",
    # Location
    "borough_clean", "city", "zip_code", "street_name",
    "latitude", "longitude", "has_geo",
    # Derived
    "is_closed", "days_open", "open_year", "open_month", "open_dow",
    # Metadata
    "_silver_at",
]

silver = silver.select(*SILVER_COLS)

## 4. Deduplicate on complaint_id

In [0]:
from pyspark.sql.window import Window

dedup_window = Window.partitionBy("complaint_id").orderBy(F.col("created_date").desc())

silver_deduped = (
    silver
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

raw_count   = silver.count()
dedup_count = silver_deduped.count()
print(f"Before dedup: {raw_count:,}")
print(f"After dedup:  {dedup_count:,}")
print(f"Duplicates removed: {raw_count - dedup_count:,}")

## 5. Write Silver Delta Table

In [0]:
(
    silver_deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("open_year", "borough_clean")
    .saveAsTable(SILVER_TABLE)
)

print(f"✓ Silver table written: {SILVER_TABLE}")
print(f"  Rows: {spark.table(SILVER_TABLE).count():,}")

## 6. Optimize & Vacuum

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (complaint_type, borough_clean)")
spark.sql(f"VACUUM {SILVER_TABLE} RETAIN 168 HOURS")  # 7 days

print("✓ OPTIMIZE + VACUUM complete.")

## 7. Silver Profile

In [0]:
SILVER_TABLE = 'civicops.silver.complaints'; 
silver_df = spark.table(SILVER_TABLE)

In [0]:
# Top complaint types
print("Top 10 Complaint Types:")
display(
    silver_df.groupBy("complaint_type")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

In [0]:
# Borough distribution
print("Complaints by Borough:")
display(
    silver_df.groupBy("borough_clean")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Resolution time stats
print("Resolution Time (days) — closed complaints only:")
display(
    silver_df.filter(F.col("is_closed"))
    .agg(
        F.round(F.avg("days_open"), 1).alias("avg_days"),
        F.percentile_approx("days_open", 0.5).alias("median_days"),
        F.percentile_approx("days_open", 0.9).alias("p90_days"),
        F.max("days_open").alias("max_days"),
    )
)
